# Reproduce the paper's figures

This notebook rebuilds all ten figures of *Grain friction breaks the separability
of granular temperature and inertial number in dense granular flow* (A. Profumo)
without LAMMPS. It uses the analysis code and derived tables in
[the repository](https://github.com/alexandermprofumo-rgb/granular-temperature-rheology) and the per-run LAMMPS logs archived on Zenodo
([10.5281/zenodo.22879491](https://doi.org/10.5281/zenodo.22879491), CC-BY-4.0,
97 MB). The figure scripts read the logs directly, so the derived CSVs alone are
not enough.

Run the cells top to bottom. In Colab the whole notebook takes a few minutes,
most of it the download.

## 1. Get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/alexandermprofumo-rgb/granular-temperature-rheology"

if os.environ.get("REPO_DIR"):                      # explicit override
    repo = Path(os.environ["REPO_DIR"]).resolve()
elif (Path.cwd().parent / "src" / "nfit.py").exists():  # opened from a checkout
    repo = Path.cwd().parent.resolve()
else:                                              # Colab or a bare download
    repo = Path("granular-temperature-rheology").resolve()
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)

SRC = repo / "src"
print("code in", SRC)

## 2. Install the pinned dependencies

The versions match `requirements.txt` and need Python 3.12 or newer. The figure
scripts run in a separate Python process, so no runtime restart is needed after
this cell.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==2.5.1", "scipy==1.18.0", "matplotlib==3.11.1"], check=True)

## 3. Download and unpack the LAMMPS logs from Zenodo

In [ ]:
import hashlib, shutil, tarfile, urllib.request

ARCHIVE_URL = "https://zenodo.org/records/22879491/files/lammps-logs-v1.1.0.tar.xz?download=1"
ARCHIVE_MD5 = "e244aae6eafdcbbf71d07ece1d05af03"
archive = repo / "lammps-logs-v1.1.0.tar.xz"

if not archive.exists():
    urllib.request.urlretrieve(ARCHIVE_URL, archive)
md5 = hashlib.md5(archive.read_bytes()).hexdigest()
assert md5 == ARCHIVE_MD5, f"checksum mismatch: {md5}"

if not (SRC / "sweep_steady2d").exists():
    with tarfile.open(archive) as t:
        t.extractall(SRC, filter="data")

# The packing drawn in Figure 1 ships in src/packing/; the schematic script
# expects it beside the matched-2D logs.
(SRC / "sweep_matched2d").mkdir(exist_ok=True)
shutil.copy(SRC / "packing" / "data.m2d_seed1", SRC / "sweep_matched2d" / "data.m2d_seed1")
print(sum(1 for _ in SRC.glob("sweep_*/log.*")), "log files unpacked")

## 4. Run the figure scripts

`make_paper_figures.py` writes Figures 2 to 10 and `make_fig_schematic.py` writes
Figure 1, all as PNG files in `src/`.

In [ ]:
env = dict(os.environ, MPLBACKEND="Agg")
for script in ["make_paper_figures.py", "make_fig_schematic.py"]:
    r = subprocess.run([sys.executable, script], cwd=SRC, env=env,
                       capture_output=True, text=True)
    print(f"--- {script} (exit {r.returncode})")
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{script} failed")

## 5. The figures

In [ ]:
from IPython.display import Image, display

### Figure 1: `fig0_setup.png`

What one run and one scan are: (a) a window of the 4000-grain 2D packing the runs start from, with the imposed shear, the barostat direction and the bath forces marked; (b) one temperature scan at grain friction 0.3.

In [ ]:
display(Image(filename=str(SRC / "fig0_setup.png"), width=800))

### Figure 2: `fig2_drift.png`

Drift of the effective friction through the measurement window at a total strain of 0.12 and at the unit strain used in the paper. This comparison fixed the protocol.

In [ ]:
display(Image(filename=str(SRC / "fig2_drift.png"), width=800))

### Figure 3: `fig3_velprof.png`

Binned streaming velocity against position at target temperature 1e-3, with the affine profile that is subtracted before granular temperature is computed.

In [ ]:
display(Image(filename=str(SRC / "fig3_velprof.png"), width=800))

### Figure 4: `fig4_rheology.png`

Effective friction against inertial number with the thermostat removed, so that temperature is generated by the shear alone.

In [ ]:
display(Image(filename=str(SRC / "fig4_rheology.png"), width=800))

### Figure 5: `fig5_jamming.png`

Coordination number against granular temperature for frictionless packings in 2D and 3D, with the isostatic counts marked. Heating drives the packings through the threshold the jamming cutoff excludes.

In [ ]:
display(Image(filename=str(SRC / "fig5_jamming.png"), width=800))

### Figure 6: `fig2_curvature.png`

Effective friction against granular temperature on log axes for representative grain frictions. A pure power law would be a straight line; the curvature is why n is reported as a local slope.

In [ ]:
display(Image(filename=str(SRC / "fig2_curvature.png"), width=800))

### Figure 7: `fig3_frictionless.png`

The frictionless baselines: (a) friction against temperature in 2D and 3D; (b) the measured exponent against the value a fabric-counting argument requires, which is 9 to 11 times larger.

In [ ]:
display(Image(filename=str(SRC / "fig3_frictionless.png"), width=800))

### Figure 8: `fig4_ncurve.png`

(a) The exponent n against grain friction in both dimensions; (b) the difference between dimensions under two treatments of the fitting window.

In [ ]:
display(Image(filename=str(SRC / "fig4_ncurve.png"), width=800))

### Figure 9: `fig5_channels.png`

The three contributions to n (normal force, tangential grip, geometry) against grain friction.

In [ ]:
display(Image(filename=str(SRC / "fig5_channels.png"), width=800))

### Figure 10: `fig6_chinull.png`

The channel quantities against the Coulomb-mobilized fraction chi, pooled over the friction, pressure, stiffness and tangential-stiffness scans, with one curve fitted per panel.

In [ ]:
display(Image(filename=str(SRC / "fig6_chinull.png"), width=800))

## 6. Check against the manuscript's figures

`notebooks/expected_figures.md5` holds the checksums of the PNG files used in the
manuscript. A match means the figure was rebuilt byte for byte. A mismatch on a
different platform can come from font rendering alone; compare the images by eye
before reading anything into it.

In [ ]:
expected = dict(line.split()[::-1] for line in
                (repo / "notebooks" / "expected_figures.md5").read_text().splitlines() if line.strip())
for name, want in expected.items():
    got = hashlib.md5((SRC / name).read_bytes()).hexdigest()
    print(f"{name:24s} {'match' if got == want else 'DIFFERS'}")